# Transformer 기반 한국어 챗봇

## 프로젝트 개요

한국어 질문-답변 데이터를 전처리하고, MeCab 형태소 분석과 한국어 Word2Vec 기반
Lexical Substitution으로 데이터를 증강한 뒤 Transformer Encoder-Decoder 챗봇을
학습하는 프로젝트입니다.

## 현재 확인된 데이터 결과

| 항목 | 결과 |
|---|---:|
| 원본 데이터 | 11,823개 |
| 완전 중복 제거 | 73개 |
| 전처리 후 데이터 | 11,750개 |
| 최종 증강 데이터 | 32,819개 |
| 결측값 | 0개 |
| 증강 데이터 열 | question, answer, label, source, group_id |

## 실행 전 준비

1. Google Colab에서 T4 GPU를 선택합니다.
2. Google Drive를 연결합니다.
3. 다음 파일이 존재하는지 확인합니다.

```text
/content/drive/MyDrive/transformer_chatbot/chatbot_augmented.csv
```

4. 아래 셀을 위에서부터 순서대로 실행합니다.

> 이 GitHub용 정리본은 기존 노트북의 반복 오류 셀과 실패 출력은 제거하고,
> 마지막 최종 통합 코드 중심으로 재구성했습니다.


## 0. 기본 설정


In [ ]:
# 0. 기본 설정
# ============================================================

SEED = 42

MAX_VOCAB_SIZE = 20000
MAX_QUESTION_LEN = 40
MAX_ANSWER_LEN = 42

BATCH_SIZE = 64
EPOCHS = 10

N_LAYERS = 1
D_MODEL = 368
N_HEADS = 8
D_FF = 1024
DROPOUT = 0.2
WARMUP_STEPS = 1000

# 기존 가중치와 학습 로그가 있더라도 다시 학습하려면 True
FORCE_RETRAIN = False

assert D_MODEL % N_HEADS == 0


# ============================================================


## 1. 라이브러리 준비


In [ ]:
# 1. 라이브러리 준비
# ============================================================

import sys
import subprocess
import importlib
import json
import os
import random
import re
import shutil
from pathlib import Path

try:
    from mecab import MeCab
except ModuleNotFoundError:
    print("python-mecab-ko를 설치합니다.")

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        "python-mecab-ko"
    ])

    importlib.invalidate_caches()

    from mecab import MeCab


import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from IPython.display import display


random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow 버전:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

if not tf.config.list_physical_devices("GPU"):
    print("⚠️ GPU가 보이지 않습니다.")
    print("런타임 → 런타임 유형 변경 → T4 GPU를 선택하세요.")

for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

print("✅ 라이브러리 준비 완료")


# ============================================================


## 2. 프로젝트 경로 설정


In [ ]:
# 2. 프로젝트 경로 설정
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/transformer_chatbot"
)

PROJECT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

AUGMENTED_PATH = (
    PROJECT_DIR / "chatbot_augmented.csv"
)

QUESTION_VOCAB_PATH = (
    PROJECT_DIR / "question_vocabulary.txt"
)

ANSWER_VOCAB_PATH = (
    PROJECT_DIR / "answer_vocabulary.txt"
)

BEST_WEIGHT_PATH = (
    PROJECT_DIR / "best_chatbot.weights.h5"
)

TRAINING_LOG_PATH = (
    PROJECT_DIR / "chatbot_training_log.csv"
)

LOSS_GRAPH_PATH = (
    PROJECT_DIR / "loss_graph.png"
)

ACCURACY_GRAPH_PATH = (
    PROJECT_DIR / "accuracy_graph.png"
)

TEST_RESULT_PATH = (
    PROJECT_DIR / "chatbot_test_results.csv"
)

SUMMARY_PATH = (
    PROJECT_DIR / "project_summary.txt"
)

CONFIG_PATH = (
    PROJECT_DIR / "model_config.json"
)

print("프로젝트 폴더:", PROJECT_DIR)


# ============================================================


## 3. 증강 데이터 불러오기


In [ ]:
# 3. 증강 데이터 불러오기
# ============================================================

if not AUGMENTED_PATH.exists():
    raise FileNotFoundError(
        "chatbot_augmented.csv를 찾지 못했습니다.\n"
        f"확인 경로: {AUGMENTED_PATH}"
    )

chatbot_df = pd.read_csv(
    AUGMENTED_PATH
)

required_columns = {
    "question",
    "answer",
    "source",
    "group_id"
}

if not required_columns.issubset(chatbot_df.columns):
    raise ValueError(
        "필요한 열이 없습니다.\n"
        f"현재 열: {chatbot_df.columns.tolist()}"
    )

chatbot_df = (
    chatbot_df
    .dropna(subset=["question", "answer", "group_id"])
    .drop_duplicates(subset=["question", "answer"])
    .reset_index(drop=True)
    .copy()
)

chatbot_df["question"] = (
    chatbot_df["question"].astype(str)
)

chatbot_df["answer"] = (
    chatbot_df["answer"].astype(str)
)

print("\n[증강 데이터]")
print("데이터 크기:", chatbot_df.shape)
print("열 이름:", chatbot_df.columns.tolist())
print("결측값:", chatbot_df.isnull().sum().sum())

print("\n[데이터 종류]")
print(chatbot_df["source"].value_counts())

assert len(chatbot_df) >= 30000
assert chatbot_df[["question", "answer"]].isnull().sum().sum() == 0

print("✅ 증강 데이터 정상 확인")


# ============================================================


## 4. 답변 시작·종료 토큰 추가


In [ ]:
# 4. 답변 시작·종료 토큰 추가
# ============================================================

START_TOKEN = "<start>"
END_TOKEN = "<end>"

chatbot_df["answer_with_tokens"] = (
    START_TOKEN
    + " "
    + chatbot_df["answer"]
    + " "
    + END_TOKEN
)

print("\n[답변 토큰 예시]")
print(chatbot_df["answer_with_tokens"].iloc[0])


# ============================================================


## 5. Train / Validation 그룹 분리


In [ ]:
# 5. Train / Validation 그룹 분리
# ============================================================

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.1,
    random_state=SEED
)

train_indices, val_indices = next(
    splitter.split(
        chatbot_df,
        groups=chatbot_df["group_id"]
    )
)

train_df = (
    chatbot_df
    .iloc[train_indices]
    .reset_index(drop=True)
)

val_df = (
    chatbot_df
    .iloc[val_indices]
    .reset_index(drop=True)
)

train_groups = set(train_df["group_id"])
val_groups = set(val_df["group_id"])

overlap_groups = (
    train_groups.intersection(val_groups)
)

assert len(overlap_groups) == 0

print("\n[데이터 분리]")
print("전체:", len(chatbot_df))
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("중복 그룹:", len(overlap_groups))

print("✅ 원본과 증강본의 데이터 누수 방지 완료")


# ============================================================


## 6. TextVectorization 생성


In [ ]:
# 6. TextVectorization 생성
# ============================================================

tf.keras.backend.clear_session()

question_vectorizer = (
    tf.keras.layers.TextVectorization(
        max_tokens=MAX_VOCAB_SIZE,
        standardize=None,
        split="whitespace",
        output_mode="int",
        output_sequence_length=MAX_QUESTION_LEN
    )
)

answer_vectorizer = (
    tf.keras.layers.TextVectorization(
        max_tokens=MAX_VOCAB_SIZE,
        standardize=None,
        split="whitespace",
        output_mode="int",
        output_sequence_length=MAX_ANSWER_LEN
    )
)


# ============================================================


## 7. 기존 어휘사전 복원 또는 새로 생성


In [ ]:
# 7. 기존 어휘사전 복원 또는 새로 생성
# ============================================================

saved_vocab_exists = (
    QUESTION_VOCAB_PATH.exists()
    and ANSWER_VOCAB_PATH.exists()
)

if saved_vocab_exists and not FORCE_RETRAIN:
    question_saved_vocab = (
        QUESTION_VOCAB_PATH
        .read_text(encoding="utf-8")
        .splitlines()
    )

    answer_saved_vocab = (
        ANSWER_VOCAB_PATH
        .read_text(encoding="utf-8")
        .splitlines()
    )

    question_vectorizer.set_vocabulary(
        question_saved_vocab
    )

    answer_vectorizer.set_vocabulary(
        answer_saved_vocab
    )

    print("\n✅ 기존 어휘사전을 복원했습니다.")

else:
    question_text_dataset = (
        tf.data.Dataset
        .from_tensor_slices(
            train_df["question"].values
        )
        .batch(512)
    )

    answer_text_dataset = (
        tf.data.Dataset
        .from_tensor_slices(
            train_df["answer_with_tokens"].values
        )
        .batch(512)
    )

    question_vectorizer.adapt(
        question_text_dataset
    )

    answer_vectorizer.adapt(
        answer_text_dataset
    )

    # 0번 Padding, 1번 OOV를 제외하고 저장
    question_vocab_to_save = (
        question_vectorizer
        .get_vocabulary()[2:]
    )

    answer_vocab_to_save = (
        answer_vectorizer
        .get_vocabulary()[2:]
    )

    QUESTION_VOCAB_PATH.write_text(
        "\n".join(question_vocab_to_save),
        encoding="utf-8"
    )

    ANSWER_VOCAB_PATH.write_text(
        "\n".join(answer_vocab_to_save),
        encoding="utf-8"
    )

    print("\n✅ 새로운 어휘사전을 생성하고 저장했습니다.")


question_vocab = (
    question_vectorizer.get_vocabulary()
)

answer_vocab = (
    answer_vectorizer.get_vocabulary()
)

INPUT_VOCAB_SIZE = len(question_vocab)
TARGET_VOCAB_SIZE = len(answer_vocab)

if START_TOKEN not in answer_vocab:
    raise ValueError("<start> 토큰이 답변 어휘에 없습니다.")

if END_TOKEN not in answer_vocab:
    raise ValueError("<end> 토큰이 답변 어휘에 없습니다.")

START_ID = answer_vocab.index(START_TOKEN)
END_ID = answer_vocab.index(END_TOKEN)

print("\n[어휘 정보]")
print("질문 어휘 수:", INPUT_VOCAB_SIZE)
print("답변 어휘 수:", TARGET_VOCAB_SIZE)
print("<start> ID:", START_ID)
print("<end> ID:", END_ID)


# ============================================================


## 8. 정수 벡터화


In [ ]:
# 8. 정수 벡터화
# ============================================================

enc_train = question_vectorizer(
    tf.constant(train_df["question"].values)
)

answer_train = answer_vectorizer(
    tf.constant(train_df["answer_with_tokens"].values)
)

enc_val = question_vectorizer(
    tf.constant(val_df["question"].values)
)

answer_val = answer_vectorizer(
    tf.constant(val_df["answer_with_tokens"].values)
)

# Teacher Forcing
dec_train = answer_train[:, :-1]
target_train = answer_train[:, 1:]

dec_val = answer_val[:, :-1]
target_val = answer_val[:, 1:]

print("\n[벡터 Shape]")
print("Encoder Train:", enc_train.shape)
print("Decoder Train:", dec_train.shape)
print("Target Train:", target_train.shape)

print("Encoder Validation:", enc_val.shape)
print("Decoder Validation:", dec_val.shape)
print("Target Validation:", target_val.shape)


# ============================================================


## 9. tf.data 데이터셋 생성


In [ ]:
# 9. tf.data 데이터셋 생성
# ============================================================

train_dataset = (
    tf.data.Dataset
    .from_tensor_slices(
        (
            (enc_train, dec_train),
            target_train
        )
    )
    .shuffle(
        buffer_size=min(len(train_df), 10000),
        seed=SEED,
        reshuffle_each_iteration=True
    )
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_dataset = (
    tf.data.Dataset
    .from_tensor_slices(
        (
            (enc_val, dec_val),
            target_val
        )
    )
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

for (
    encoder_batch,
    decoder_batch
), target_batch in train_dataset.take(1):

    print("\n[배치 Shape]")
    print("Encoder:", encoder_batch.shape)
    print("Decoder:", decoder_batch.shape)
    print("Target:", target_batch.shape)

print("✅ train_dataset과 val_dataset 생성 완료")


# ============================================================


## 10. 위치 인코딩


In [ ]:
# 10. 위치 인코딩
# ============================================================

def positional_encoding(max_position, d_model):
    positions = (
        np.arange(max_position)[:, np.newaxis]
    )

    dimensions = (
        np.arange(d_model)[np.newaxis, :]
    )

    angle_rates = (
        1
        / np.power(
            10000,
            (2 * (dimensions // 2))
            / np.float32(d_model)
        )
    )

    angle_radians = positions * angle_rates

    angle_radians[:, 0::2] = np.sin(
        angle_radians[:, 0::2]
    )

    angle_radians[:, 1::2] = np.cos(
        angle_radians[:, 1::2]
    )

    return tf.constant(
        angle_radians[np.newaxis, ...],
        dtype=tf.float32
    )


class PositionalEncoding(tf.keras.layers.Layer):
    def __init__(self, max_position, d_model):
        super().__init__()

        self.position_values = positional_encoding(
            max_position,
            d_model
        )

    def call(self, inputs):
        sequence_length = tf.shape(inputs)[1]

        return (
            inputs
            + self.position_values[
                :, :sequence_length, :
            ]
        )


# ============================================================


## 11. Attention Mask


In [ ]:
# 11. Attention Mask
# ============================================================

def create_padding_mask(token_ids):
    # 실제 단어=True, Padding=False
    mask = tf.not_equal(token_ids, 0)

    # (batch, 1, source_length)
    return mask[:, tf.newaxis, :]


def create_look_ahead_mask(token_ids):
    sequence_length = tf.shape(token_ids)[1]

    causal_mask = tf.linalg.band_part(
        tf.ones(
            (sequence_length, sequence_length),
            dtype=tf.bool
        ),
        -1,
        0
    )

    padding_mask = (
        tf.not_equal(token_ids, 0)
        [:, tf.newaxis, :]
    )

    # (batch, target_length, source_length)
    return tf.logical_and(
        causal_mask[tf.newaxis, :, :],
        padding_mask
    )


# ============================================================


## 12. Encoder Layer


In [ ]:
# 12. Encoder Layer
# ============================================================

class EncoderLayer(tf.keras.layers.Layer):
    def __init__(
        self,
        d_model,
        n_heads,
        d_ff,
        dropout
    ):
        super().__init__()

        self.self_attention = (
            tf.keras.layers.MultiHeadAttention(
                num_heads=n_heads,
                key_dim=d_model // n_heads,
                dropout=dropout
            )
        )

        self.feed_forward = tf.keras.Sequential([
            tf.keras.layers.Dense(
                d_ff,
                activation="relu"
            ),
            tf.keras.layers.Dense(d_model)
        ])

        self.norm_1 = (
            tf.keras.layers.LayerNormalization(
                epsilon=1e-6
            )
        )

        self.norm_2 = (
            tf.keras.layers.LayerNormalization(
                epsilon=1e-6
            )
        )

        self.dropout_1 = (
            tf.keras.layers.Dropout(dropout)
        )

        self.dropout_2 = (
            tf.keras.layers.Dropout(dropout)
        )

    def call(
        self,
        inputs,
        padding_mask,
        training=False
    ):
        attention_output = self.self_attention(
            query=inputs,
            value=inputs,
            key=inputs,
            attention_mask=padding_mask,
            training=training
        )

        attention_output = self.dropout_1(
            attention_output,
            training=training
        )

        output_1 = self.norm_1(
            inputs + attention_output
        )

        ff_output = self.feed_forward(
            output_1,
            training=training
        )

        ff_output = self.dropout_2(
            ff_output,
            training=training
        )

        return self.norm_2(
            output_1 + ff_output
        )


# ============================================================


## 13. Decoder Layer


In [ ]:
# 13. Decoder Layer
# ============================================================

class DecoderLayer(tf.keras.layers.Layer):
    def __init__(
        self,
        d_model,
        n_heads,
        d_ff,
        dropout
    ):
        super().__init__()

        self.masked_self_attention = (
            tf.keras.layers.MultiHeadAttention(
                num_heads=n_heads,
                key_dim=d_model // n_heads,
                dropout=dropout
            )
        )

        self.cross_attention = (
            tf.keras.layers.MultiHeadAttention(
                num_heads=n_heads,
                key_dim=d_model // n_heads,
                dropout=dropout
            )
        )

        self.feed_forward = tf.keras.Sequential([
            tf.keras.layers.Dense(
                d_ff,
                activation="relu"
            ),
            tf.keras.layers.Dense(d_model)
        ])

        self.norm_1 = (
            tf.keras.layers.LayerNormalization(
                epsilon=1e-6
            )
        )

        self.norm_2 = (
            tf.keras.layers.LayerNormalization(
                epsilon=1e-6
            )
        )

        self.norm_3 = (
            tf.keras.layers.LayerNormalization(
                epsilon=1e-6
            )
        )

        self.dropout_1 = (
            tf.keras.layers.Dropout(dropout)
        )

        self.dropout_2 = (
            tf.keras.layers.Dropout(dropout)
        )

        self.dropout_3 = (
            tf.keras.layers.Dropout(dropout)
        )

    def call(
        self,
        inputs,
        encoder_output,
        look_ahead_mask,
        encoder_padding_mask,
        training=False
    ):
        attention_1 = self.masked_self_attention(
            query=inputs,
            value=inputs,
            key=inputs,
            attention_mask=look_ahead_mask,
            training=training
        )

        attention_1 = self.dropout_1(
            attention_1,
            training=training
        )

        output_1 = self.norm_1(
            inputs + attention_1
        )

        attention_2 = self.cross_attention(
            query=output_1,
            value=encoder_output,
            key=encoder_output,
            attention_mask=encoder_padding_mask,
            training=training
        )

        attention_2 = self.dropout_2(
            attention_2,
            training=training
        )

        output_2 = self.norm_2(
            output_1 + attention_2
        )

        ff_output = self.feed_forward(
            output_2,
            training=training
        )

        ff_output = self.dropout_3(
            ff_output,
            training=training
        )

        return self.norm_3(
            output_2 + ff_output
        )


# ============================================================


## 14. 전체 Transformer 모델


In [ ]:
# 14. 전체 Transformer 모델
# ============================================================

class TransformerChatbot(tf.keras.Model):
    def __init__(
        self,
        input_vocab_size,
        target_vocab_size,
        n_layers,
        d_model,
        n_heads,
        d_ff,
        dropout,
        max_position=128
    ):
        super().__init__()

        self.d_model = d_model

        self.encoder_embedding = (
            tf.keras.layers.Embedding(
                input_vocab_size,
                d_model
            )
        )

        self.decoder_embedding = (
            tf.keras.layers.Embedding(
                target_vocab_size,
                d_model
            )
        )

        self.encoder_position = PositionalEncoding(
            max_position,
            d_model
        )

        self.decoder_position = PositionalEncoding(
            max_position,
            d_model
        )

        self.encoder_dropout = (
            tf.keras.layers.Dropout(dropout)
        )

        self.decoder_dropout = (
            tf.keras.layers.Dropout(dropout)
        )

        self.encoder_layers = [
            EncoderLayer(
                d_model,
                n_heads,
                d_ff,
                dropout
            )
            for _ in range(n_layers)
        ]

        self.decoder_layers = [
            DecoderLayer(
                d_model,
                n_heads,
                d_ff,
                dropout
            )
            for _ in range(n_layers)
        ]

        self.final_dense = (
            tf.keras.layers.Dense(
                target_vocab_size,
                dtype="float32"
            )
        )

    def call(self, inputs, training=False):
        encoder_tokens, decoder_tokens = inputs

        encoder_padding_mask = (
            create_padding_mask(
                encoder_tokens
            )
        )

        look_ahead_mask = (
            create_look_ahead_mask(
                decoder_tokens
            )
        )

        # Encoder
        encoder_output = self.encoder_embedding(
            encoder_tokens
        )

        encoder_output *= tf.math.sqrt(
            tf.cast(self.d_model, tf.float32)
        )

        encoder_output = self.encoder_position(
            encoder_output
        )

        encoder_output = self.encoder_dropout(
            encoder_output,
            training=training
        )

        for encoder_layer in self.encoder_layers:
            encoder_output = encoder_layer(
                encoder_output,
                padding_mask=encoder_padding_mask,
                training=training
            )

        # Decoder
        decoder_output = self.decoder_embedding(
            decoder_tokens
        )

        decoder_output *= tf.math.sqrt(
            tf.cast(self.d_model, tf.float32)
        )

        decoder_output = self.decoder_position(
            decoder_output
        )

        decoder_output = self.decoder_dropout(
            decoder_output,
            training=training
        )

        for decoder_layer in self.decoder_layers:
            decoder_output = decoder_layer(
                decoder_output,
                encoder_output=encoder_output,
                look_ahead_mask=look_ahead_mask,
                encoder_padding_mask=encoder_padding_mask,
                training=training
            )

        return self.final_dense(decoder_output)


# ============================================================


## 15. Loss와 Accuracy


In [ ]:
# 15. Loss와 Accuracy
# ============================================================

loss_object = (
    tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits=True,
        reduction="none"
    )
)


def masked_loss(y_true, y_pred):
    loss = loss_object(y_true, y_pred)

    mask = tf.cast(
        tf.not_equal(y_true, 0),
        loss.dtype
    )

    loss *= mask

    return (
        tf.reduce_sum(loss)
        / tf.maximum(
            tf.reduce_sum(mask),
            1.0
        )
    )


def masked_accuracy(y_true, y_pred):
    predicted_ids = tf.argmax(
        y_pred,
        axis=-1,
        output_type=y_true.dtype
    )

    correct = tf.equal(
        y_true,
        predicted_ids
    )

    mask = tf.not_equal(y_true, 0)

    correct = tf.logical_and(
        correct,
        mask
    )

    correct = tf.cast(
        correct,
        tf.float32
    )

    mask = tf.cast(
        mask,
        tf.float32
    )

    return (
        tf.reduce_sum(correct)
        / tf.maximum(
            tf.reduce_sum(mask),
            1.0
        )
    )


# ============================================================


## 16. Warmup Learning Rate


In [ ]:
# 16. Warmup Learning Rate
# ============================================================

class TransformerSchedule(
    tf.keras.optimizers.schedules.LearningRateSchedule
):
    def __init__(
        self,
        d_model,
        warmup_steps=1000
    ):
        super().__init__()

        self.d_model_value = d_model
        self.warmup_steps_value = warmup_steps

        self.d_model = tf.cast(
            d_model,
            tf.float32
        )

        self.warmup_steps = tf.cast(
            warmup_steps,
            tf.float32
        )

    def __call__(self, step):
        step = tf.cast(step, tf.float32)

        inverse_sqrt = tf.math.rsqrt(
            tf.maximum(step, 1.0)
        )

        warmup = (
            step
            * tf.math.pow(
                self.warmup_steps,
                -1.5
            )
        )

        return (
            tf.math.rsqrt(self.d_model)
            * tf.math.minimum(
                inverse_sqrt,
                warmup
            )
        )

    def get_config(self):
        return {
            "d_model": self.d_model_value,
            "warmup_steps": self.warmup_steps_value
        }


# ============================================================


## 17. 모델 생성 및 컴파일


In [ ]:
# 17. 모델 생성 및 컴파일
# ============================================================

transformer = TransformerChatbot(
    input_vocab_size=INPUT_VOCAB_SIZE,
    target_vocab_size=TARGET_VOCAB_SIZE,
    n_layers=N_LAYERS,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    d_ff=D_FF,
    dropout=DROPOUT,
    max_position=128
)

learning_rate = TransformerSchedule(
    D_MODEL,
    warmup_steps=WARMUP_STEPS
)

optimizer = tf.keras.optimizers.Adam(
    learning_rate=learning_rate,
    beta_1=0.9,
    beta_2=0.98,
    epsilon=1e-9,
    clipnorm=1.0
)

transformer.compile(
    optimizer=optimizer,
    loss=masked_loss,
    metrics=[masked_accuracy]
)

# 모델 Build 및 출력 Shape 검사
for (
    sample_encoder,
    sample_decoder
), sample_target in train_dataset.take(1):

    sample_logits = transformer(
        (sample_encoder, sample_decoder),
        training=False
    )

    print("\n[모델 검사]")
    print("Encoder:", sample_encoder.shape)
    print("Decoder:", sample_decoder.shape)
    print("Target:", sample_target.shape)
    print("모델 출력:", sample_logits.shape)

print("✅ transformer 생성 및 compile 완료")


# ============================================================


## 18. Callback 설정


In [ ]:
# 18. Callback 설정
# ============================================================

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(BEST_WEIGHT_PATH),
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    ),

    tf.keras.callbacks.CSVLogger(
        filename=str(TRAINING_LOG_PATH),
        append=False
    ),

    tf.keras.callbacks.TerminateOnNaN()
]

print("✅ callbacks 생성 완료")


# ============================================================


## 19. 모델 학습 또는 기존 모델 복원


In [ ]:
# 19. 모델 학습 또는 기존 모델 복원
# ============================================================

can_restore = (
    BEST_WEIGHT_PATH.exists()
    and TRAINING_LOG_PATH.exists()
    and QUESTION_VOCAB_PATH.exists()
    and ANSWER_VOCAB_PATH.exists()
    and not FORCE_RETRAIN
)

if can_restore:
    transformer.load_weights(
        str(BEST_WEIGHT_PATH)
    )

    print("\n✅ 기존 최적 가중치를 복원했습니다.")

else:
    print("\n" + "=" * 70)
    print("Transformer 학습 시작")
    print("=" * 70)

    history = transformer.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=EPOCHS,
        callbacks=callbacks
    )

    transformer.load_weights(
        str(BEST_WEIGHT_PATH)
    )

    print("\n✅ 학습 및 최적 가중치 복원 완료")


# ============================================================


## 20. 학습 로그 확인


In [ ]:
# 20. 학습 로그 확인
# ============================================================

if not TRAINING_LOG_PATH.exists():
    raise FileNotFoundError(
        "학습 로그가 생성되지 않았습니다."
    )

training_log = pd.read_csv(
    TRAINING_LOG_PATH
)

display(training_log)

best_index = (
    training_log["val_loss"].idxmin()
)

best_epoch = int(best_index) + 1

best_train_loss = float(
    training_log.loc[best_index, "loss"]
)

best_val_loss = float(
    training_log.loc[best_index, "val_loss"]
)

print("\n[최적 학습 결과]")
print("실제 학습 Epoch:", len(training_log))
print("최적 Epoch:", best_epoch)
print("최적 Train Loss:", round(best_train_loss, 4))
print("최적 Validation Loss:", round(best_val_loss, 4))
print(
    "Loss 차이:",
    round(best_val_loss - best_train_loss, 4)
)


# ============================================================


## 21. Loss 그래프 저장


In [ ]:
# 21. Loss 그래프 저장
# ============================================================

epoch_numbers = np.arange(
    1,
    len(training_log) + 1
)

plt.figure(figsize=(8, 5))

plt.plot(
    epoch_numbers,
    training_log["loss"],
    marker="o",
    label="Train Loss"
)

plt.plot(
    epoch_numbers,
    training_log["val_loss"],
    marker="o",
    label="Validation Loss"
)

plt.axvline(
    best_epoch,
    linestyle="--",
    label=f"Best Epoch: {best_epoch}"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Transformer Chatbot Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    LOSS_GRAPH_PATH,
    dpi=150,
    bbox_inches="tight"
)

plt.show()

print("✅ Loss 그래프 저장:", LOSS_GRAPH_PATH)


# ============================================================


## 22. Accuracy 그래프 저장


In [ ]:
# 22. Accuracy 그래프 저장
# ============================================================

train_acc_col = None
val_acc_col = None

for column in [
    "masked_accuracy",
    "accuracy"
]:
    if column in training_log.columns:
        train_acc_col = column
        break

for column in [
    "val_masked_accuracy",
    "val_accuracy"
]:
    if column in training_log.columns:
        val_acc_col = column
        break

if train_acc_col and val_acc_col:
    plt.figure(figsize=(8, 5))

    plt.plot(
        epoch_numbers,
        training_log[train_acc_col],
        marker="o",
        label="Train Accuracy"
    )

    plt.plot(
        epoch_numbers,
        training_log[val_acc_col],
        marker="o",
        label="Validation Accuracy"
    )

    plt.axvline(
        best_epoch,
        linestyle="--",
        label=f"Best Epoch: {best_epoch}"
    )

    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Transformer Chatbot Accuracy")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()

    plt.savefig(
        ACCURACY_GRAPH_PATH,
        dpi=150,
        bbox_inches="tight"
    )

    plt.show()

    print(
        "✅ Accuracy 그래프 저장:",
        ACCURACY_GRAPH_PATH
    )

else:
    print(
        "⚠️ Accuracy 열을 찾지 못했습니다:",
        training_log.columns.tolist()
    )


# ============================================================


## 23. 질문 전처리 및 답변 생성 함수


In [ ]:
# 23. 질문 전처리 및 답변 생성 함수
# ============================================================

mecab = MeCab()


def preprocess_sentence(sentence):
    sentence = str(sentence).strip()

    sentence = re.sub(
        r"([?.!,])",
        r" \1 ",
        sentence
    )

    sentence = re.sub(
        r"[^가-힣a-zA-Z0-9?.!,]+",
        " ",
        sentence
    )

    sentence = re.sub(
        r"\s+",
        " ",
        sentence
    )

    return sentence.strip()


def format_generated_tokens(tokens):
    text = " ".join(tokens)

    # 문장부호 앞 공백 제거
    text = re.sub(
        r"\s+([?.!,])",
        r"\1",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


def prepare_question(sentence):
    cleaned_sentence = preprocess_sentence(
        sentence
    )

    question_tokens = mecab.morphs(
        cleaned_sentence
    )

    tokenized_sentence = " ".join(
        question_tokens
    )

    encoder_input = question_vectorizer(
        tf.constant([tokenized_sentence])
    )

    return tokenized_sentence, encoder_input


def generate_answer(
    sentence,
    max_decoder_len=MAX_ANSWER_LEN - 1
):
    tokenized_question, encoder_input = (
        prepare_question(sentence)
    )

    decoder_array = np.zeros(
        (1, max_decoder_len),
        dtype=np.int32
    )

    decoder_array[0, 0] = START_ID

    generated_ids = []

    for step in range(max_decoder_len):
        decoder_input = tf.constant(
            decoder_array,
            dtype=tf.int32
        )

        predictions = transformer(
            (encoder_input, decoder_input),
            training=False
        )

        next_token_logits = (
            predictions[0, step, :]
            .numpy()
            .copy()
        )

        # Padding, OOV, <start> 선택 금지
        for banned_id in [
            0,
            1,
            START_ID
        ]:
            if 0 <= banned_id < len(next_token_logits):
                next_token_logits[banned_id] = -1e9

        next_token_id = int(
            np.argmax(next_token_logits)
        )

        if next_token_id == END_ID:
            break

        generated_ids.append(
            next_token_id
        )

        if step + 1 < max_decoder_len:
            decoder_array[
                0,
                step + 1
            ] = next_token_id

    generated_tokens = [
        answer_vocab[token_id]
        for token_id in generated_ids
        if 0 <= token_id < len(answer_vocab)
    ]

    generated_answer = format_generated_tokens(
        generated_tokens
    )

    return {
        "question": sentence,
        "tokenized_question": tokenized_question,
        "answer_tokens": generated_tokens,
        "answer": generated_answer
    }


print("✅ generate_answer 함수 생성 완료")


# ============================================================


## 24. 프로젝트 예문 테스트


In [ ]:
# 24. 프로젝트 예문 테스트
# ============================================================

test_questions = [
    "지루하다, 놀러가고 싶어.",
    "오늘 일찍 일어났더니 피곤하다.",
    "간만에 여자친구랑 데이트 하기로 했어.",
    "집에 있는데도 집에 가고 싶어.",
    "오늘 기분이 너무 좋아.",
    "친구와 싸워서 속상해.",
    "시험을 잘 볼 수 있을까?",
    "요즘 너무 외로워."
]

chatbot_results = []

print("\n" + "=" * 70)
print("챗봇 답변 결과")
print("=" * 70)

for question in test_questions:
    result = generate_answer(question)

    chatbot_results.append({
        "question": question,
        "tokenized_question": (
            result["tokenized_question"]
        ),
        "generated_answer": (
            result["answer"]
        )
    })

    print("\n질문:", question)
    print(
        "형태소:",
        result["tokenized_question"]
    )
    print("답변:", result["answer"])
    print("-" * 70)


result_df = pd.DataFrame(
    chatbot_results
)

result_df.to_csv(
    TEST_RESULT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\n✅ 답변 결과 저장:", TEST_RESULT_PATH)


# ============================================================


## 25. 모델 설정 및 프로젝트 요약 저장


In [ ]:
# 25. 모델 설정 및 프로젝트 요약 저장
# ============================================================

model_config = {
    "data_count": len(chatbot_df),
    "train_count": len(train_df),
    "validation_count": len(val_df),
    "input_vocab_size": INPUT_VOCAB_SIZE,
    "target_vocab_size": TARGET_VOCAB_SIZE,
    "max_question_len": MAX_QUESTION_LEN,
    "max_answer_len": MAX_ANSWER_LEN,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "n_layers": N_LAYERS,
    "d_model": D_MODEL,
    "n_heads": N_HEADS,
    "head_dimension": D_MODEL // N_HEADS,
    "d_ff": D_FF,
    "dropout": DROPOUT,
    "warmup_steps": WARMUP_STEPS,
    "best_epoch": best_epoch,
    "best_train_loss": best_train_loss,
    "best_validation_loss": best_val_loss
}

CONFIG_PATH.write_text(
    json.dumps(
        model_config,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

summary_text = f"""
Transformer 기반 한국어 챗봇 프로젝트
=====================================

1. 데이터 전처리
- 원본 데이터: 11,823개
- 완전 중복 제거: 73개
- 전처리 후 데이터: 11,750개
- Word2Vec 증강 데이터: {len(chatbot_df):,}개
- 결측값: 0개

2. 데이터 분리
- Train: {len(train_df):,}개
- Validation: {len(val_df):,}개
- 그룹 중복: 0개

3. Transformer 하이퍼파라미터
- Encoder Layer: {N_LAYERS}
- Decoder Layer: {N_LAYERS}
- d_model: {D_MODEL}
- Attention Heads: {N_HEADS}
- Head Dimension: {D_MODEL // N_HEADS}
- Feed Forward Dimension: {D_FF}
- Dropout: {DROPOUT}
- Warmup Steps: {WARMUP_STEPS}
- Batch Size: {BATCH_SIZE}

4. 학습 결과
- 실제 학습 Epoch: {len(training_log)}
- 최적 Epoch: {best_epoch}
- 최적 Train Loss: {best_train_loss:.4f}
- 최적 Validation Loss: {best_val_loss:.4f}
- Loss Gap: {(best_val_loss - best_train_loss):.4f}

5. 과적합 방지
- Word2Vec 데이터 증강
- 그룹 단위 Train/Validation 분리
- Dropout 0.2
- Warmup Learning Rate
- Gradient Clipping
- Early Stopping
- Best Model Checkpoint
- Padding을 제외한 Loss 및 Accuracy
""".strip()

SUMMARY_PATH.write_text(
    summary_text,
    encoding="utf-8"
)

print("\n" + summary_text)
print("\n✅ 프로젝트 요약 저장 완료")


# ============================================================


## 26. GitHub 제출 폴더 생성


In [ ]:
# 26. GitHub 제출 폴더 생성
# ============================================================

GITHUB_DIR = (
    PROJECT_DIR / "github_submission"
)

RESULTS_DIR = (
    GITHUB_DIR / "results"
)

DATA_DIR = (
    GITHUB_DIR / "data"
)

GITHUB_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# 결과 파일 복사
result_files = [
    LOSS_GRAPH_PATH,
    ACCURACY_GRAPH_PATH,
    TRAINING_LOG_PATH,
    TEST_RESULT_PATH,
    SUMMARY_PATH,
    CONFIG_PATH
]

for source_path in result_files:
    if source_path.exists():
        shutil.copy2(
            source_path,
            RESULTS_DIR / source_path.name
        )


## 3. 26MB 증강 데이터도 GitHub 제출 폴더에 포함


In [ ]:
# 3.26MB 증강 데이터도 GitHub 제출 폴더에 포함
shutil.copy2(
    AUGMENTED_PATH,
    DATA_DIR / AUGMENTED_PATH.name
)


# ============================================================


## 27. README 자동 생성


In [ ]:
# 27. README 자동 생성
# ============================================================

answer_rows = []

for _, row in result_df.iterrows():
    q = str(row["question"]).replace("|", "｜")
    a = str(row["generated_answer"]).replace("|", "｜")

    answer_rows.append(
        f"| {q} | {a} |"
    )

answer_table = "\n".join(answer_rows)

readme_text = f"""# Transformer 기반 한국어 챗봇

## 프로젝트 개요

한국어 질문을 입력받아 답변을 생성하는
Transformer Encoder-Decoder 기반 챗봇 프로젝트입니다.

ChatbotData를 정제하고 MeCab으로 형태소 분석한 뒤,
한국어 Word2Vec 기반 Lexical Substitution으로
학습 데이터를 증강했습니다.

## 데이터 처리 결과

| 항목 | 결과 |
|---|---:|
| 원본 데이터 | 11,823개 |
| 완전 중복 제거 | 73개 |
| 전처리 후 데이터 | 11,750개 |
| 최종 증강 데이터 | {len(chatbot_df):,}개 |
| 결측값 | 0개 |
| Train 데이터 | {len(train_df):,}개 |
| Validation 데이터 | {len(val_df):,}개 |

## 전체 진행 과정

1. 결측값 및 중복 확인
2. 문장부호·특수문자 정제
3. MeCab 형태소 분석
4. Word2Vec Lexical Substitution
5. 그룹 단위 Train/Validation 분리
6. TextVectorization
7. Teacher Forcing 데이터 구성
8. Transformer Encoder-Decoder 학습
9. 실제 챗봇 답변 생성

## 배치 Shape

- Encoder Input: `(64, 40)`
- Decoder Input: `(64, 41)`
- Target: `(64, 41)`

## Transformer 설정

| 항목 | 값 |
|---|---:|
| Encoder Layer | {N_LAYERS} |
| Decoder Layer | {N_LAYERS} |
| d_model | {D_MODEL} |
| Attention Heads | {N_HEADS} |
| Head Dimension | {D_MODEL // N_HEADS} |
| Feed Forward Dimension | {D_FF} |
| Dropout | {DROPOUT} |
| Warmup Steps | {WARMUP_STEPS} |
| Batch Size | {BATCH_SIZE} |

## 학습 결과

| 항목 | 결과 |
|---|---:|
| 실제 학습 Epoch | {len(training_log)} |
| 최적 Epoch | {best_epoch} |
| 최적 Train Loss | {best_train_loss:.4f} |
| 최적 Validation Loss | {best_val_loss:.4f} |

### Loss

![Loss Graph](results/loss_graph.png)

### Accuracy

![Accuracy Graph](results/accuracy_graph.png)

## 챗봇 답변 사례

| 질문 | 생성된 답변 |
|---|---|
{answer_table}

## 과적합 방지

- Word2Vec 데이터 증강
- 그룹 단위 Train/Validation 분리
- Dropout 0.2
- Warmup Learning Rate
- Gradient Clipping
- Early Stopping
- Best Model Checkpoint

## 프로젝트 구조

- `README.md`
- `RETROSPECTIVE.md`
- `requirements.txt`
- `.gitignore`
- `data/chatbot_augmented.csv`
- `results/loss_graph.png`
- `results/accuracy_graph.png`
- `results/chatbot_training_log.csv`
- `results/chatbot_test_results.csv`
- `results/project_summary.txt`

## 한계와 개선 방향

1. Word2Vec 유사어가 항상 동의어는 아닙니다.
2. Greedy Decoding은 답변의 다양성이 제한될 수 있습니다.
3. SentencePiece와 Beam Search를 적용할 수 있습니다.
4. 감정 label을 활용한 조건부 생성 모델로 확장할 수 있습니다.
"""

(GITHUB_DIR / "README.md").write_text(
    readme_text,
    encoding="utf-8"
)


# ============================================================


## 28. 회고문 자동 생성


In [ ]:
# 28. 회고문 자동 생성
# ============================================================

retrospective_text = """# Transformer 한국어 챗봇 프로젝트 회고

## 잘한 점

- 질문과 답변을 하나의 쌍으로 처리해 순서가 어긋나지 않도록 했다.
- 원본과 증강본에 group_id를 부여해 데이터 누수를 방지했다.
- Dropout, Warmup, Early Stopping, Checkpoint를 적용했다.
- 중간 데이터와 학습 결과를 Google Drive에 저장했다.

## 어려웠던 점

### MeCab 설치

기존 설치 방식에서 오류가 발생하여
python-mecab-ko 패키지를 사용해 해결했다.

### Word2Vec 호환성

오래된 Word2Vec 파일을 Gensim 4의
KeyedVectors 형식으로 변환해야 했다.

### 런타임 초기화

Colab 런타임이 초기화되면서 변수와 임시 파일이 사라졌다.
이후 데이터, 어휘사전, 가중치와 로그를
Google Drive에 저장하도록 개선했다.

## 새롭게 이해한 점

- 모델 구조뿐 아니라 데이터 품질과 파일 관리가 중요하다.
- Word2Vec 유사어는 반드시 동의어가 아니다.
- Teacher Forcing에서는 Decoder 입력과 Target을 한 칸 이동한다.
- 챗봇 평가는 Loss와 BLEU뿐 아니라 실제 답변도 확인해야 한다.

## 아쉬운 점

- 모든 증강 문장을 사람이 직접 검수하지 못했다.
- Greedy Decoding만 사용했다.
- 다양한 하이퍼파라미터 비교 실험을 수행하지 못했다.
- 감정 label을 모델 입력에 활용하지 못했다.

## 개선 방향

- SentencePiece 기반 Subword Tokenization
- Beam Search 또는 Top-k Sampling
- 문맥 기반 데이터 증강
- Transformer Ablation Study
- 감정 label 기반 조건부 답변 생성

## 최종 회고

이번 프로젝트를 통해 데이터 전처리부터 증강,
Transformer 학습, 결과 저장과 실제 답변 생성까지
하나의 자연어 처리 프로젝트 전체 흐름을 경험했다.

오류를 해결하면서 각 단계의 입력과 출력을 확인하고
중간 산출물을 영구 저장하는 습관의 중요성을 배웠다.
"""

(GITHUB_DIR / "RETROSPECTIVE.md").write_text(
    retrospective_text,
    encoding="utf-8"
)


# ============================================================


## 29. requirements와 gitignore 생성


In [ ]:
# 29. requirements와 gitignore 생성
# ============================================================

requirements_text = """tensorflow
numpy
pandas
scikit-learn
python-mecab-ko
matplotlib
"""

(GITHUB_DIR / "requirements.txt").write_text(
    requirements_text,
    encoding="utf-8"
)

gitignore_text = """# 대용량 모델 및 임베딩
*.h5
*.weights.h5
*.kv
*.bin
*.zip
*.tsv

# Python
__pycache__/
*.pyc

# Jupyter
.ipynb_checkpoints/

# 환경 및 인증정보
.env
*.key
*.pem

# 운영체제
.DS_Store
Thumbs.db
"""

(GITHUB_DIR / ".gitignore").write_text(
    gitignore_text,
    encoding="utf-8"
)


# ============================================================


## 30. 최종 결과 확인


In [ ]:
# 30. 최종 결과 확인
# ============================================================

final_files = [
    AUGMENTED_PATH,
    QUESTION_VOCAB_PATH,
    ANSWER_VOCAB_PATH,
    BEST_WEIGHT_PATH,
    TRAINING_LOG_PATH,
    LOSS_GRAPH_PATH,
    ACCURACY_GRAPH_PATH,
    TEST_RESULT_PATH,
    SUMMARY_PATH,
    CONFIG_PATH,
    GITHUB_DIR / "README.md",
    GITHUB_DIR / "RETROSPECTIVE.md",
    GITHUB_DIR / "requirements.txt",
    GITHUB_DIR / ".gitignore",
    DATA_DIR / "chatbot_augmented.csv"
]

print("\n" + "=" * 70)
print("최종 파일 확인")
print("=" * 70)

all_complete = True

for path in final_files:
    exists = path.exists()

    print(
        "✅" if exists else "❌",
        path
    )

    if not exists:
        all_complete = False

print("\n전체 완료 여부:", all_complete)
print("GitHub 제출 폴더:", GITHUB_DIR)

print("\n🎉 Transformer 챗봇 프로젝트 최종 처리 완료")


## 제출 전 확인사항

이 정리본에서 모델 학습을 실행하면 다음 파일이 Google Drive에 생성됩니다.

```text
best_chatbot.weights.h5
chatbot_training_log.csv
loss_graph.png
accuracy_graph.png
chatbot_test_results.csv
project_summary.txt
model_config.json
github_submission/
```

GitHub에는 가중치와 Word2Vec 대용량 파일을 올리지 않고, 노트북·README·회고·그래프·로그만 올리는 것을 권장합니다.
